# Extended Data Figure 5 — inverse folding

Reproduces the panels of this figure. Each cell runs a released producer and shows its output.

**Default level is 1** — redraw from a table shipped in the Zenodo deposit. No model, no GPU;
seconds on a laptop. Cells that need a GPU or a checkpoint are marked and left commented out.

Run from the `peint-paper` repository, or set `PEINT_PAPER_REPO` to point at it.

In [ ]:
import os, sys, subprocess, pathlib
# Point these at your checkout. PAPER_REPO is the peint-paper repository; PEINT_REPO the model
# library. Everything below runs from the paper repository root.
PAPER_REPO = pathlib.Path(os.environ.get("PEINT_PAPER_REPO", "..")).resolve()
os.environ.setdefault("PEINT_PAPER_PEINT_REPO", str(PAPER_REPO.parent / "peint"))
os.environ["PEINT_PAPER_LOCAL_DATA_ONLY"] = "1"
os.chdir(PAPER_REPO)
sys.path.insert(0, str(PAPER_REPO))

import pandas as pd, numpy as np
import matplotlib.pyplot as plt

# Inline display needs IPython. These notebooks are also executed headlessly in CI, where it is
# absent, so fall back to printing rather than failing the whole notebook on the import.
try:
    from IPython.display import display, Image
    _INLINE = True
except ImportError:
    _INLINE = False
    def display(x): print(x)
    def Image(filename=None, width=None): return f"[figure written to {filename}]"

_last_run = [0.0]   # when the most recent producer started, so show() can spot stale images

def run(module, *args):
    """Run a panel producer and show its output."""
    import time
    cmd = [sys.executable, "-m", module, *args]
    print("$", " ".join(cmd[2:]))
    _last_run[0] = time.time()
    r = subprocess.run(cmd, capture_output=True, text=True)
    tail = [l for l in r.stdout.split("\n") if l.strip()][-15:]
    print("\n".join(tail))
    if r.returncode:
        print(f"\n  *** FAILED (exit {r.returncode}) ***")
        print("  " + r.stderr.strip().split("\n")[-1][:400])
    return r.returncode == 0

def show(*names, w=760):
    """Display panel images from figures/output.

    A panel file older than the producer call above it is left over from an earlier run --
    displaying it silently would show you a figure that does not match the code you just ran,
    so say so rather than pretending it is fresh.
    """
    for n in names:
        p = PAPER_REPO / "figures" / "output" / (n + ".png")
        if p.exists():
            if p.stat().st_mtime < _last_run[0]:
                print(f"  *** STALE: {n}.png predates the run above -- it did not regenerate. "
                      f"The image below is from an earlier run; do not read numbers off it. ***")
            display(Image(filename=str(p), width=w))
        elif (p.with_suffix(".pdf")).exists():
            print(f"  {n}.pdf written (no PNG to display inline)")
        else:
            print("  not found:", n)

def compare(label, ours, printed, tol=0.05):
    """Print our value against the manuscript's."""
    ok = "MATCH" if abs(ours - printed) <= tol else "CHECK"
    print(f"  {label:<28} ours {ours:>8.3f}   manuscript {printed:>8.3f}   {ok}")

## 5d, 5e — ESM-IF likelihood and sequence recovery

Level 2. Needs the trRosetta structures (5d) and the `r1_omegafold` role (5e, 12.4 GB).
Budget roughly an hour on one GPU; **check that a GPU is actually visible first** — the
script falls back to CPU silently, which turns an hour into most of a day.

In [ ]:
import torch
print("CUDA visible:", torch.cuda.is_available())
assert torch.cuda.is_available(), "run this on a GPU node - the CPU path takes ~9.5 h"
run("benchmarks.esmif_validation", "--approach", "both")

## 5b, 5c — not reproducible from the deposit

These need an AlphaFold2-with-MSA table that is not in the deposit (a small file, well
under a megabyte). Tracked as **P1-15**; nothing to run until it ships.